In [ ]:
from datasets import load_dataset

dataset = load_dataset("detection-datasets/coco", split="val[:6]")
print(f"Loaded {len(dataset)} images")
print(f"First 5 = context (labeled), Last 1 = test (agent will annotate)")

In [ ]:
import json

COCO_CATEGORIES = {
    0: "person", 1: "bicycle", 2: "car", 3: "motorcycle", 4: "airplane",
    5: "bus", 6: "train", 7: "truck", 8: "boat", 9: "traffic light",
    10: "fire hydrant", 11: "stop sign", 12: "parking meter", 13: "bench",
    14: "bird", 15: "cat", 16: "dog", 17: "horse", 18: "sheep", 19: "cow",
    20: "elephant", 21: "bear", 22: "zebra", 23: "giraffe", 24: "backpack",
    25: "umbrella", 26: "handbag", 27: "tie", 28: "suitcase", 29: "frisbee",
    30: "skis", 31: "snowboard", 32: "sports ball", 33: "kite",
    34: "baseball bat", 35: "baseball glove", 36: "skateboard", 37: "surfboard",
    38: "tennis racket", 39: "bottle", 40: "wine glass", 41: "cup", 42: "fork",
    43: "knife", 44: "spoon", 45: "bowl", 46: "banana", 47: "apple",
    48: "sandwich", 49: "orange", 50: "broccoli", 51: "carrot", 52: "hot dog",
    53: "pizza", 54: "donut", 55: "cake", 56: "chair", 57: "couch",
    58: "potted plant", 59: "bed", 60: "dining table", 61: "toilet", 62: "tv",
    63: "laptop", 64: "mouse", 65: "remote", 66: "keyboard", 67: "cell phone",
    68: "microwave", 69: "oven", 70: "toaster", 71: "sink", 72: "refrigerator",
    73: "book", 74: "clock", 75: "vase", 76: "scissors", 77: "teddy bear",
    78: "hair drier", 79: "toothbrush"
}

context_examples = []
for i in range(5):
    entry = dataset[i]
    objects = entry["objects"]
    
    annotations = []
    for j in range(len(objects["bbox"])):
        x_min, y_min, w, h = objects["bbox"][j]
        label = COCO_CATEGORIES.get(objects["category"][j], "unknown")
        annotations.append({
            "bbox": [round(x_min, 2), round(y_min, 2), round(x_min + w, 2), round(y_min + h, 2)],
            "label": label,
            "confidence": 1.0
        })
    
    context_examples.append(annotations)
    print(f"Image {i+1}: {len(annotations)} objects → {[a['label'] for a in annotations]}")

print(f"\nTotal: {sum(len(a) for a in context_examples)} annotations across 5 images")

In [ ]:
import base64
import io

def image_to_base64(pil_image):
    buffer = io.BytesIO()
    pil_image.convert("RGB").save(buffer, format="JPEG", quality=90)
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

context_images_b64 = []
for i in range(5):
    context_images_b64.append(image_to_base64(dataset[i]["image"]))

test_image_b64 = image_to_base64(dataset[5]["image"])

print(f"Converted 5 context images to base64")
print(f"Converted 1 test image to base64")

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

print("Loading Grounding DINO...")
model_id = "IDEA-Research/grounding-dino-tiny"
processor = AutoProcessor.from_pretrained(model_id)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id)
print("Grounding DINO loaded!")

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatAnthropic(
    model="claude-sonnet-4-20250514",
    api_key=""
    max_tokens=4096
)
print("Claude loaded!")

In [ ]:
def tide_annotate(new_image, context_images_b64, context_examples, llm, processor, dino_model):
    """
    Full TIDE pipeline:
    1. Claude Vision analyzes context → returns labels to detect
    2. Grounding DINO takes labels + image → returns precise bboxes
    """

    # STEP 1: Ask Claude what to detect
    print("Step 1: Asking Claude what objects to detect...")

    content = []
    for i in range(len(context_images_b64)):
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{context_images_b64[i]}"}
        })
        content.append({
            "type": "text",
            "text": f"Example {i+1} annotations:\n{json.dumps(context_examples[i], indent=2)}"
        })

    new_image_b64 = image_to_base64(new_image)
    content.append({
        "type": "image_url",
        "image_url": {"url": f"data:image/jpeg;base64,{new_image_b64}"}
    })
    content.append({
        "type": "text",
        "text": 'This is the NEW image. Based on the annotation style from the examples, what objects should be labeled? Return ONLY a JSON object like: {"labels": ["person", "car", "dog"]}'
    })

    from pydantic import BaseModel, Field
    from typing import List

    class DetectLabels(BaseModel):
      labels: List[str] = Field(description="List of object lables to detect")

    structured_llm = llm.with_structured_output(DetectLabels)

    response = structured_llm.invoke([
        SystemMessage(content="You analyze images and determine what objects to detect based on annotation examples. Return ONLY valid JSON with a labels array."),
        HumanMessage(content=content)
    ])
    labels = response.labels

    print(f"  Claude's raw response content: {labels}") # Added this line for debugging
    print(f"  Claude says detect: {labels}")

    # STEP 2: Run Grounding DINO with those labels
    print("Step 2: Running Grounding DINO for precise bounding boxes...")

    labels_text = ". ".join(labels) + "."

    inputs = processor(images=new_image, text=labels_text, return_tensors="pt")
    with torch.no_grad():
        outputs = dino_model(**inputs)

    results = processor.post_process_grounded_object_detection(
        outputs,
        inputs["input_ids"],
        threshold=0.3,
        target_sizes=[new_image.size[::-1]]
    )[0]

    # STEP 3: Format results
    annotations = []
    for box, score, label in zip(results["boxes"], results["scores"], results["text_labels"]):
        x_min, y_min, x_max, y_max = box.tolist()
        annotations.append({
            "bbox": [round(x_min, 2), round(y_min, 2), round(x_max, 2), round(y_max, 2)],
            "label": label,
            "confidence": round(score.item(), 3)
        })

    print(f"  Found {len(annotations)} objects")
    return annotations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Run the full pipeline
test_image = dataset[5]["image"]
annotations = tide_annotate(test_image, context_images_b64, context_examples, llm, processor, dino_model)

# Print results
print("\n=== FINAL ANNOTATIONS ===")
for ann in annotations:
    print(f"  {ann['label']}: bbox={ann['bbox']}, confidence={ann['confidence']}")

# Visualize
fig, ax = plt.subplots(1, figsize=(10, 8))
ax.imshow(test_image)

for ann in annotations:
    x_min, y_min, x_max, y_max = ann["bbox"]
    rect = patches.Rectangle(
        (x_min, y_min), x_max - x_min, y_max - y_min,
        linewidth=2, edgecolor='red', facecolor='none'
    )
    ax.add_patch(rect)
    ax.text(x_min, y_min - 5, f'{ann["label"]} ({ann["confidence"]})',
        color='white', fontsize=10, bbox=dict(facecolor='red', alpha=0.7))

plt.axis('off')
plt.title("TIDE Agent - Full Pipeline (Claude + Grounding DINO)")
plt.show()

In [ ]:
# Load more images to test on a complex one
dataset_big = load_dataset("detection-datasets/coco", split="val[6:10]")

# Pick one and run the pipeline
for i in range(len(dataset_big)):
    print(f"\n{'='*50}")
    print(f"Testing image {i+1}")
    test_img = dataset_big[i]["image"]
    anns = tide_annotate(test_img, context_images_b64, context_examples, llm, processor, dino_model)
    
    print(f"\nAnnotations:")
    for ann in anns:
        print(f"  {ann['label']}: confidence={ann['confidence']}")
    
    fig, ax = plt.subplots(1, figsize=(10, 8))
    ax.imshow(test_img)
    for ann in anns:
        x_min, y_min, x_max, y_max = ann["bbox"]
        rect = patches.Rectangle(
            (x_min, y_min), x_max - x_min, y_max - y_min,
            linewidth=2, edgecolor='red', facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x_min, y_min - 5, f'{ann["label"]} ({ann["confidence"]})',
            color='white', fontsize=9, bbox=dict(facecolor='red', alpha=0.7))
    plt.axis('off')
    plt.title(f"TIDE Agent - Test Image {i+1}")
    plt.show()